# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane: Refresh / Content Opportunity Scoring (provisional, confirmable through Week 4).**

I'm choosing this lane as my starting point because the starter repo already ships a working, runnable end-to-end example of it (`scripts/01`–`05`), so I can learn the full workflow — prepare features, build a transparent baseline, train a model, evaluate, export — by reading and running real code before I try to extend it myself. It also produces the clearest story for a beginner: one row is one page, the output is a ranked list, and there's an obvious real-world action (a reviewer opens the top page first) with an obvious cost if the ranking is wrong (wasted reviewer time, or a genuinely declining page left unreviewed). I'll confirm or swap this lane by the end of Week 4 once I've done more EDA.

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Research question:** Which pages, out of a client's content inventory, should a content reviewer look at first for a possible refresh?

**Unit of analysis:** one row = one content page (`content_id`), aggregated over its trailing 90-day window.

**Decision improved:** where a content team spends its limited review time this cycle — today that decision is either ad-hoc or based on a hand-built rule; this project tests whether a ranked, evidence-backed queue does better.

**Output:** a ranked list (a "review queue") of pages, each with a score and reason codes explaining *why* it's in the queue (e.g. "declining_with_demand", "stale_visible_page").

**Action someone takes:** a content reviewer opens the top N pages on the list first and decides whether to refresh, expand, or leave each one — the model/score never edits or publishes anything itself; a human always makes the final call.

**Cost of a wrong call — two directions, not symmetric:**
- *False positive* (page ranked high but didn't actually need review): a reviewer spends ~10–15 minutes checking a page that turns out fine. Annoying, low cost.
- *False negative* (a genuinely declining page never makes the queue): the page keeps losing visibility/traffic for weeks or months with nobody noticing. Higher cost, because it compounds silently over time.

Because false negatives are more expensive here, the eventual model's threshold should lean toward catching more true declines, even at the price of a few extra false positives — something I'll revisit once I actually train a model.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [ ]:
import pandas as pd

# Load the starter dataset (relative path from work/notebooks/ up to repo root)
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# 1) How big is the inventory we'd be ranking, and across how many clients?
n_rows, n_cols = df.shape
n_clients = df["client_id"].nunique()
print(f"Rows (pages): {n_rows:,} | Columns: {n_cols} | Distinct clients: {n_clients}")

# 2) How many pages are currently *declining* by the simple trend rule?
#    (trend_direction == 'down' is the same signal the label is built from later --
#    we're just reading it here, not using it as a feature.)
declining_share = (df["trend_direction"] == "down").mean()
print(f"Share of pages currently trending down: {declining_share:.1%}")

# 3) Are these declining pages actually worth reviewing -- do they still have real search demand?
#    (a page with 0 impressions declining from 0 isn't a useful queue candidate)
visible_and_declining = df[(df["trend_direction"] == "down") & (df["impressions_90d"] >= 100)]
print(f"Declining pages with >=100 impressions/90d (real candidates): {len(visible_and_declining):,}")
print(f"...that's {len(visible_and_declining) / n_rows:.1%} of the whole starter dataset")

**Reading these numbers (expected, per the repo's documented data dictionary — confirm the printed output matches when you run this):**

- ~30,000 pages across 32 clients is a real, if modest, inventory — enough to build and validate a ranking, with room for a client-holdout split (see the leakage note below).
- A large share of pages are trending down by the simple rule — this alone isn't proof the lane is useful (a lot of 'down' could be low-value noise), which is exactly why the third number matters.
- Restricting to pages that are both declining *and* have real search demand (>=100 impressions/90d) shows the 'down' pages aren't just empty/low-traffic noise — there's a meaningful subset worth a reviewer's time, which is the actual review-queue problem this lane is meant to solve.

**Beginner trap I'm flagging for myself:** `trend_direction` (and `trend_pct`) are the exact columns the eventual label (`is_declining_label`) is built from. I read them here only to describe the dataset — I will **never** use them as model features later, or the model would just be copying its own answer (label leakage).

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What I CAN say, if the analysis holds up:**
- "We *observed* that pages with X characteristics were more likely to be trending down in this 90-day window."
- "The ranked queue is *directional* — it prioritizes review, it doesn't guarantee an outcome."
- "This is a *decision-support* tool: it ranks candidates for a human reviewer; it does not publish or edit content itself."
- "On this starter slice, a learned model beat the hand-written baseline at Precision@50 (a documented, reproducible result) — evidence that ranking can add value over a fixed rule, on this dataset."

**What I will NOT say, ever:**
- That I *predicted* or *reverse-engineered* Google's ranking algorithm.
- That refreshing a page *caused* a recovery — I have no experiment (no A/B test, no randomized holdout of refreshed vs. non-refreshed pages), only observational data.
- That `trend_direction == "down"` is proof of a *real* decline rather than seasonality, consolidation (a sibling page absorbing traffic), or plain noise — I have not ruled those out yet.
- That any hashed `client_id` / `content_id` maps back to a real client, domain, or URL.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.